# Residual U-Net rainfall retrieval

13 TB channels + 9 selected ERA5 channels; log1p rainfall normalization fitted on training data; custom LSM/SD training selection; random file-level train/validation/test split; 50 epochs; batch size 32; learning rate 1e-4; **SWD weight 0 (MSE-only)**.

The optional SWD compares full flattened patches as vectors, projects them, then sorts across the batch. It is computed in log-rainfall space, with fixed projection directions and no rainy-pixel mask. This is a Res-UNet notebook, not the conditional diffusion model.

The random file split does not guarantee temporal or orbit independence. Keep the supplied split for reproduction; use independently prepared orbit/year splits for experiments requiring that independence. The loader holds data in RAM, including normalized copies; start with smaller split counts if needed.

<p style="color: green;"><strong>Note: The same ResUNet predictions are used for all diffusion models compared in this study. ResUNet is trained without SWD regularization. This paper focuses on improving the diffusion model and the final rainfall retrievals obtained by correcting the ResUNet predictions.</strong></p>

## 1. Dependencies
Run this cell once. PyTorch is imported from the runtime; the remaining packages are installed only if missing.

In [1]:
import importlib.util
import subprocess
import sys
missing = [p for p in ['numpy', 'matplotlib', 'tqdm'] if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Install PyTorch for your runtime before continuing.')

## 2. Imports and reproducibility
The seed controls file selection, loader shuffling, and initialization. GPU execution may still vary slightly.

In [2]:
import os
import json
import random
import csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
colors_rain = ['gainsboro', 'seagreen', 'green', 'yellow', 'darkorange', 'red']
cmap_rate_rain = LinearSegmentedColormap.from_list('mycmap_rain', colors_rain)

PyTorch: 2.4.1
Device: NVIDIA A40


## 3. Optional Google Drive mount
Set `USE_GOOGLE_DRIVE=True` to access data and save outputs on Drive. Leave it false for local data under `/content`. The original HPC path is not available automatically in Colab.

In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 4. Configuration
Edit paths and sample counts here. Defaults retain the supplied training settings. For an initial small run, try 100 training files, 20 validation files, 20 test files, and 1 epoch. Small runs are for checking the workflow.

`ZERO_THRESHOLD=0.1` controls evaluation event classification only; it does not mask training or SWD. The rainfall plot scale saturates at 4 mm/hr, as in the supplied code.

In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/resunet_data' if USE_GOOGLE_DRIVE else '/content/resunet_data')
PATCHES_DIR = DATA_ROOT / 'grl_dpr_patches_32X32_UNet'
OUTPUT_DIR = str(Path('/content/drive/MyDrive/resunet_swd_outputs' if USE_GOOGLE_DRIVE else '/content/resunet_swd_outputs'))
NUM_EPOCHS = 50
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
LAMBDA_SWD = 0.0
NUM_PROJECTIONS = 16
ZERO_THRESHOLD = 0.1
LSM_IDX = 25
LSM_THRESHOLD = 0.5
SD_IDX = 21
LOW_NO_SD_CAP_TRAIN = 25000
HIGH_NO_SD_CAP_TRAIN = 25000
TRAIN_FILE_COUNT = 169500
VAL_FILE_COUNT = 1000
TEST_FILE_COUNT = 1000
EVALUATION_SAMPLES = None  # None evaluates every loaded test patch and saves one plot per patch.
INFERENCE_BATCH_SIZE = 32
MODEL_LABEL = 'Residual U-Net' if LAMBDA_SWD == 0 else 'Residual U-Net + SWD'
os.makedirs(OUTPUT_DIR, exist_ok=True)
BEST_MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_resunet_model.pth')
HISTORY_PLOT_PATH = os.path.join(OUTPUT_DIR, 'resunet_training_history.png')
COMPARISON_DIR = os.path.join(OUTPUT_DIR, 'comparison_plots')
os.makedirs(COMPARISON_DIR, exist_ok=True)

## 5. Residual convolution block
Two convolutions with a shortcut connection.

In [ ]:
class ResidualConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ResidualConvBlock, self).__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2   = nn.BatchNorm2d(out_channels)

        if in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = out + identity
        out = self.relu(out)

        return out

## 6. Residual U-Net architecture
TB and ERA5 are concatenated into 22 input channels. Four pooling stages require compatible spatial dimensions; the supplied patches are 32 × 32.

In [ ]:
class ResidualUNet(nn.Module):
    def __init__(self, tb_channels=13, era5_channels=9, base_channels=64):
        super(ResidualUNet, self).__init__()

        in_channels = tb_channels + era5_channels

        self.enc1 = ResidualConvBlock(in_channels, base_channels)
        self.enc2 = ResidualConvBlock(base_channels, base_channels * 2)
        self.enc3 = ResidualConvBlock(base_channels * 2, base_channels * 4)
        self.enc4 = ResidualConvBlock(base_channels * 4, base_channels * 8)

        self.bottleneck = ResidualConvBlock(base_channels * 8, base_channels * 16)

        self.up4 = nn.ConvTranspose2d(base_channels * 16, base_channels * 8, kernel_size=2, stride=2)
        self.dec4 = ResidualConvBlock(base_channels * 16, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(base_channels * 8, base_channels * 4, kernel_size=2, stride=2)
        self.dec3 = ResidualConvBlock(base_channels * 8, base_channels * 4)

        self.up2 = nn.ConvTranspose2d(base_channels * 4, base_channels * 2, kernel_size=2, stride=2)
        self.dec2 = ResidualConvBlock(base_channels * 4, base_channels * 2)

        self.up1 = nn.ConvTranspose2d(base_channels * 2, base_channels, kernel_size=2, stride=2)
        self.dec1 = ResidualConvBlock(base_channels * 2, base_channels)

        self.final_conv = nn.Conv2d(base_channels, 1, kernel_size=1)
        self.pool = nn.MaxPool2d(2)

    def forward(self, tb_features, era5_features):
        x = torch.cat([tb_features, era5_features], dim=1)

        enc1 = self.enc1(x)
        p1 = self.pool(enc1)

        enc2 = self.enc2(p1)
        p2 = self.pool(enc2)

        enc3 = self.enc3(p2)
        p3 = self.pool(enc3)

        enc4 = self.enc4(p3)
        p4 = self.pool(enc4)

        bottleneck = self.bottleneck(p4)

        d4 = self.up4(bottleneck)
        d4 = torch.cat([d4, enc4], dim=1)
        d4 = self.dec4(d4)

        d3 = self.up3(d4)
        d3 = torch.cat([d3, enc3], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        d2 = torch.cat([d2, enc2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, enc1], dim=1)
        d1 = self.dec1(d1)

        output = self.final_conv(d1)
        return output

## 7. Training-only normalization
Input statistics use all training pixels. Rainfall log statistics use positive training rainfall. Validation/test reuse the fitted statistics. Predictions are converted back to mm/hr and clipped at zero.

In [ ]:
class DataProcessor:
    def __init__(self, zero_threshold=0.2):
        self.tb_mean = None
        self.tb_std = None
        self.era5_mean = None
        self.era5_std = None
        self.rain_mean = None
        self.rain_std = None
        self.zero_threshold = zero_threshold

    def fit(self, tb_data, era5_data, rain_data):
        self.tb_mean = tb_data.mean(dim=(0, 2, 3), keepdim=True)
        self.tb_std = tb_data.std(dim=(0, 2, 3), keepdim=True)

        self.era5_mean = era5_data.mean(dim=(0, 2, 3), keepdim=True)
        self.era5_std = era5_data.std(dim=(0, 2, 3), keepdim=True)

        rain_positive = rain_data[rain_data > 0]
        if len(rain_positive) > 0:
            rain_log = torch.log1p(rain_positive)
            self.rain_mean = rain_log.mean()
            self.rain_std = rain_log.std() if rain_log.numel() > 1 else torch.ones_like(self.rain_mean)
        else:
            self.rain_mean = torch.tensor(0.0)
            self.rain_std = torch.tensor(1.0)

    def normalize_tb(self, tb_data):
        return (tb_data - self.tb_mean) / (self.tb_std + 1e-8)

    def normalize_era5(self, era5_data):
        return (era5_data - self.era5_mean) / (self.era5_std + 1e-8)

    def normalize_rain(self, rain_data):
        rain_log = torch.log1p(rain_data.clamp(min=0))
        return (rain_log - self.rain_mean) / (self.rain_std + 1e-8)

    def denormalize_rain(self, rain_norm):
        rain_log = rain_norm * self.rain_std + self.rain_mean
        rain_physical = torch.expm1(rain_log)
        rain_physical = torch.clamp(rain_physical, min=0)
        return rain_physical

    def normalized_to_log_space(self, rain_norm):
        return rain_norm * self.rain_std + self.rain_mean

## 8. MSE and optional SWD loss
Total loss = MSE + lambda × SWD. SWD remains computed and logged when lambda is zero, but contributes no training gradient.

In [ ]:
class CombinedMSESWDLoss(nn.Module):
    def __init__(self, processor, lambda_swd=0.02, num_projections=16, projection_seed=1234):
        super().__init__()
        self.processor = processor
        self.lambda_swd = lambda_swd
        self.num_projections = num_projections
        self.projection_seed = projection_seed
        self.mse = nn.MSELoss()

    def sliced_wasserstein_distance(self, pred_log, target_log):
        B = pred_log.shape[0]
        pred_flat = pred_log.view(B, -1)
        target_flat = target_log.view(B, -1)

        D = pred_flat.shape[1]
        device = pred_flat.device

        gen = torch.Generator(device=device)
        gen.manual_seed(self.projection_seed + D)

        projections = torch.randn(
            self.num_projections, D,
            generator=gen,
            device=device
        )
        projections = projections / (torch.norm(projections, dim=1, keepdim=True) + 1e-8)

        pred_proj = pred_flat @ projections.T
        target_proj = target_flat @ projections.T

        pred_proj_sorted, _ = torch.sort(pred_proj, dim=0)
        target_proj_sorted, _ = torch.sort(target_proj, dim=0)

        swd = torch.mean((pred_proj_sorted - target_proj_sorted) ** 2)
        return swd

    def forward(self, pred_norm, target_norm):
        mse_loss = self.mse(pred_norm, target_norm)

        pred_log = self.processor.normalized_to_log_space(pred_norm)
        target_log = self.processor.normalized_to_log_space(target_norm)

        swd_loss = self.sliced_wasserstein_distance(pred_log, target_log)

        total_loss = mse_loss + self.lambda_swd * swd_loss
        return total_loss, mse_loss.detach(), swd_loss.detach()

## 9. Training data loader
Keep all patches with any positive SD; cap each no-SD group at 25,000 using mean LSM below/above 0.5. These are the supplied rules; verify that channel 21 represents the intended SD field in your dataset.

In [ ]:
def load_train_data_custom(
    patches_dir,
    file_list=None,
    low_no_sd_cap=25000,
    high_no_sd_cap=25000,
    lsm_idx=25,
    lsm_threshold=0.5,
    sd_idx=21,
    verbose_every=500
):
    """
    Train loading rule:
    - keep only up to low_no_sd_cap from low_lsm + no_sd
    - keep all low_lsm + has_sd
    - keep only up to high_no_sd_cap from high_lsm + no_sd
    - keep all high_lsm + has_sd
    """

    if file_list is None:
        all_files = sorted([f for f in os.listdir(patches_dir) if f.endswith('.npy')])
    else:
        all_files = list(file_list)

    all_files = list(all_files)
    np.random.shuffle(all_files)

    print("=" * 90)
    print("TRAIN DATA LOADING WITH CUSTOM FILTERING")
    print("=" * 90)
    print(f"Candidate files      : {len(all_files)}")
    print(f"low_no_sd cap        : {low_no_sd_cap}")
    print(f"high_no_sd cap       : {high_no_sd_cap}")
    print(f"LSM index            : {lsm_idx}")
    print(f"LSM threshold        : {lsm_threshold}")
    print(f"SD index             : {sd_idx}")
    print("=" * 90)

    tb_low_no_sd,   era5_low_no_sd,   rain_low_no_sd   = [], [], []
    tb_low_has_sd,  era5_low_has_sd,  rain_low_has_sd  = [], [], []
    tb_high_no_sd,  era5_high_no_sd,  rain_high_no_sd  = [], [], []
    tb_high_has_sd, era5_high_has_sd, rain_high_has_sd = [], [], []

    scanned_files = 0
    invalid_shape = 0
    nan_skipped = 0
    extreme_rain_skipped = 0
    failed_load = 0

    true_low_no_sd_total = 0
    true_low_has_sd_total = 0
    true_high_no_sd_total = 0
    true_high_has_sd_total = 0

    pbar = tqdm(all_files, desc="Scanning train files", ncols=170)

    for fname in pbar:
        scanned_files += 1

        try:
            patch = np.load(os.path.join(patches_dir, fname))
        except Exception:
            failed_load += 1
            continue

        if patch.shape != (32, 32, 31):
            invalid_shape += 1
            continue

        try:
            tb   = patch[:, :, :13]
            era5 = patch[:, :, [13, 14, 15, 16, 17, 21, 22, 23, 25]]
            rain = patch[:, :, -1:]
            sd   = patch[:, :, sd_idx]
            lsm  = patch[:, :, lsm_idx]
        except Exception:
            failed_load += 1
            continue

        if np.isnan(tb).any() or np.isnan(era5).any() or np.isnan(rain).any() or np.isnan(sd).any() or np.isnan(lsm).any():
            nan_skipped += 1
            continue

        if rain.max() > 500:
            extreme_rain_skipped += 1
            continue

        mean_lsm = lsm.mean()
        has_sd = np.any(sd > 0)

        if mean_lsm < lsm_threshold:
            if has_sd:
                true_low_has_sd_total += 1
                tb_low_has_sd.append(tb)
                era5_low_has_sd.append(era5)
                rain_low_has_sd.append(rain)
            else:
                true_low_no_sd_total += 1
                if len(tb_low_no_sd) < low_no_sd_cap:
                    tb_low_no_sd.append(tb)
                    era5_low_no_sd.append(era5)
                    rain_low_no_sd.append(rain)
        else:
            if has_sd:
                true_high_has_sd_total += 1
                tb_high_has_sd.append(tb)
                era5_high_has_sd.append(era5)
                rain_high_has_sd.append(rain)
            else:
                true_high_no_sd_total += 1
                if len(tb_high_no_sd) < high_no_sd_cap:
                    tb_high_no_sd.append(tb)
                    era5_high_no_sd.append(era5)
                    rain_high_no_sd.append(rain)

        pbar.set_postfix({
            "low_no":  f"{len(tb_low_no_sd)}/{low_no_sd_cap}",
            "low_sd":  f"{len(tb_low_has_sd)}",
            "high_no": f"{len(tb_high_no_sd)}/{high_no_sd_cap}",
            "high_sd": f"{len(tb_high_has_sd)}"
        })

        if scanned_files % verbose_every == 0:
            print("\n" + "-" * 90)
            print(f"Scanned train files: {scanned_files}")
            print(f"  kept low_no_sd   : {len(tb_low_no_sd)}/{low_no_sd_cap}")
            print(f"  kept low_has_sd  : {len(tb_low_has_sd)}")
            print(f"  kept high_no_sd  : {len(tb_high_no_sd)}/{high_no_sd_cap}")
            print(f"  kept high_has_sd : {len(tb_high_has_sd)}")
            print("-" * 90)

    print("\n" + "=" * 90)
    print("TRAIN LOADING SUMMARY")
    print("=" * 90)
    print(f"Scanned files                : {scanned_files}")
    print(f"Invalid shape skipped        : {invalid_shape}")
    print(f"NaN skipped                  : {nan_skipped}")
    print(f"Extreme rain skipped (>500)  : {extreme_rain_skipped}")
    print(f"Failed load/indexing         : {failed_load}")
    print("-" * 90)
    print("True available counts:")
    print(f"  true low_no_sd   : {true_low_no_sd_total}")
    print(f"  true low_has_sd  : {true_low_has_sd_total}")
    print(f"  true high_no_sd  : {true_high_no_sd_total}")
    print(f"  true high_has_sd : {true_high_has_sd_total}")
    print("-" * 90)
    print("Kept counts:")
    print(f"  kept low_no_sd   : {len(tb_low_no_sd)}")
    print(f"  kept low_has_sd  : {len(tb_low_has_sd)}")
    print(f"  kept high_no_sd  : {len(tb_high_no_sd)}")
    print(f"  kept high_has_sd : {len(tb_high_has_sd)}")

    tb_list = tb_low_no_sd + tb_low_has_sd + tb_high_no_sd + tb_high_has_sd
    era5_list = era5_low_no_sd + era5_low_has_sd + era5_high_no_sd + era5_high_has_sd
    rain_list = rain_low_no_sd + rain_low_has_sd + rain_high_no_sd + rain_high_has_sd

    total_loaded = len(tb_list)
    print("-" * 90)
    print(f"Total kept for train before shuffle: {total_loaded}")

    if total_loaded == 0:
        raise ValueError("No valid train samples found!")

    combined = list(zip(tb_list, era5_list, rain_list))
    np.random.shuffle(combined)
    tb_list, era5_list, rain_list = zip(*combined)

    tb_array = np.array(tb_list).transpose(0, 3, 1, 2)
    era5_array = np.array(era5_list).transpose(0, 3, 1, 2)
    rain_array = np.array(rain_list).transpose(0, 3, 1, 2)

    tb_tensor = torch.FloatTensor(tb_array)
    era5_tensor = torch.FloatTensor(era5_array)
    rain_tensor = torch.FloatTensor(rain_array)

    print("\nFinal train tensor shapes:")
    print(f"  TB   : {tb_tensor.shape}")
    print(f"  ERA5 : {era5_tensor.shape}")
    print(f"  Rain : {rain_tensor.shape}")

    return tb_tensor, era5_tensor, rain_tensor

## 10. Validation and test loader
No LSM/SD balancing. Both loaders still skip invalid shapes, NaNs, failed reads, and rainfall maxima above 500. Expected file shape: (32, 32, 31); TB indices 0–12; ERA5 indices 13,14,15,16,17,21,22,23,25; rainfall index 30.

In [ ]:
def load_data_simple(
    patches_dir,
    file_list=None,
    num_samples=None,
    verbose_every=500
):
    """
    Simple loading for validation/test:
    - no LSM filtering
    - no SD filtering
    - just load valid files normally
    """
    if file_list is None:
        all_files = sorted([f for f in os.listdir(patches_dir) if f.endswith('.npy')])
    else:
        all_files = list(file_list)

    if num_samples is not None:
        all_files = all_files[:num_samples]

    print("=" * 90)
    print("SIMPLE DATA LOADING (NO FILTERING)")
    print("=" * 90)
    print(f"Candidate files: {len(all_files)}")
    print("=" * 90)

    tb_list, era5_list, rain_list = [], [], []

    scanned_files = 0
    invalid_shape = 0
    nan_skipped = 0
    extreme_rain_skipped = 0
    failed_load = 0

    pbar = tqdm(all_files, desc="Loading files", ncols=150)

    for fname in pbar:
        scanned_files += 1

        try:
            patch = np.load(os.path.join(patches_dir, fname))
        except Exception:
            failed_load += 1
            continue

        if patch.shape != (32, 32, 31):
            invalid_shape += 1
            continue

        try:
            tb   = patch[:, :, :13]
            era5 = patch[:, :, [13, 14, 15, 16, 17, 21, 22, 23, 25]]
            rain = patch[:, :, -1:]
        except Exception:
            failed_load += 1
            continue

        if np.isnan(tb).any() or np.isnan(era5).any() or np.isnan(rain).any():
            nan_skipped += 1
            continue

        if rain.max() > 500:
            extreme_rain_skipped += 1
            continue

        tb_list.append(tb)
        era5_list.append(era5)
        rain_list.append(rain)

        if scanned_files % verbose_every == 0:
            print(f"Scanned {scanned_files} files | kept {len(tb_list)} valid samples")

    print("\n" + "=" * 90)
    print("SIMPLE LOADING SUMMARY")
    print("=" * 90)
    print(f"Scanned files                : {scanned_files}")
    print(f"Invalid shape skipped        : {invalid_shape}")
    print(f"NaN skipped                  : {nan_skipped}")
    print(f"Extreme rain skipped (>500)  : {extreme_rain_skipped}")
    print(f"Failed load/indexing         : {failed_load}")
    print(f"Total kept                   : {len(tb_list)}")

    if len(tb_list) == 0:
        raise ValueError("No valid samples found!")

    tb_array = np.array(tb_list).transpose(0, 3, 1, 2)
    era5_array = np.array(era5_list).transpose(0, 3, 1, 2)
    rain_array = np.array(rain_list).transpose(0, 3, 1, 2)

    tb_tensor = torch.FloatTensor(tb_array)
    era5_tensor = torch.FloatTensor(era5_array)
    rain_tensor = torch.FloatTensor(rain_array)

    print("\nFinal tensor shapes:")
    print(f"  TB   : {tb_tensor.shape}")
    print(f"  ERA5 : {era5_tensor.shape}")
    print(f"  Rain : {rain_tensor.shape}")

    return tb_tensor, era5_tensor, rain_tensor

## 11. Training function
Adam optimization, ReduceLROnPlateau scheduling, sample-weighted epoch losses, and best-validation checkpoint saving.

In [ ]:
def train_resunet_model(
    train_data,
    val_data,
    num_epochs=100,
    batch_size=32,
    learning_rate=1e-4,
    lambda_swd=0.05,
    num_projections=16
):
    (tb_train, era5_train, rain_train) = train_data
    (tb_val, era5_val, rain_val) = val_data

    processor = DataProcessor(zero_threshold=ZERO_THRESHOLD)
    processor.fit(tb_train, era5_train, rain_train)

    tb_train_norm = processor.normalize_tb(tb_train)
    era5_train_norm = processor.normalize_era5(era5_train)
    rain_train_norm = processor.normalize_rain(rain_train)

    tb_val_norm = processor.normalize_tb(tb_val)
    era5_val_norm = processor.normalize_era5(era5_val)
    rain_val_norm = processor.normalize_rain(rain_val)

    train_dataset = TensorDataset(tb_train_norm, era5_train_norm, rain_train_norm)
    val_dataset = TensorDataset(tb_val_norm, era5_val_norm, rain_val_norm)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = ResidualUNet().to(device)

    criterion = CombinedMSESWDLoss(
        processor=processor,
        lambda_swd=lambda_swd,
        num_projections=num_projections,
        projection_seed=1234
    )

    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=5, factor=0.5
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_mse': [], 'val_mse': [],
        'train_swd': [], 'val_swd': []
    }

    best_val_loss = float('inf')

    print(f"Training Residual U-Net model on {device}")
    print(f"Training samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}")
    print(f"Loss = MSE + {lambda_swd} * SWD")
    print("=" * 70)

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0
        train_mse_total = 0.0
        train_swd_total = 0.0

        for tb_batch, era5_batch, rain_batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
            tb_batch = tb_batch.to(device)
            era5_batch = era5_batch.to(device)
            rain_batch = rain_batch.to(device)

            optimizer.zero_grad()
            predictions = model(tb_batch, era5_batch)
            loss, mse_part, swd_part = criterion(predictions, rain_batch)

            loss.backward()
            optimizer.step()

            bs = tb_batch.size(0)
            train_loss += loss.item() * bs
            train_mse_total += mse_part.item() * bs
            train_swd_total += swd_part.item() * bs

        model.eval()
        val_loss = 0.0
        val_mse_total = 0.0
        val_swd_total = 0.0

        with torch.no_grad():
            for tb_batch, era5_batch, rain_batch in val_loader:
                tb_batch = tb_batch.to(device)
                era5_batch = era5_batch.to(device)
                rain_batch = rain_batch.to(device)

                predictions = model(tb_batch, era5_batch)
                loss, mse_part, swd_part = criterion(predictions, rain_batch)

                bs = tb_batch.size(0)
                val_loss += loss.item() * bs
                val_mse_total += mse_part.item() * bs
                val_swd_total += swd_part.item() * bs

        train_loss_avg = train_loss / len(train_dataset)
        val_loss_avg = val_loss / len(val_dataset)
        train_mse_avg = train_mse_total / len(train_dataset)
        val_mse_avg = val_mse_total / len(val_dataset)
        train_swd_avg = train_swd_total / len(train_dataset)
        val_swd_avg = val_swd_total / len(val_dataset)

        history['train_loss'].append(train_loss_avg)
        history['val_loss'].append(val_loss_avg)
        history['train_mse'].append(train_mse_avg)
        history['val_mse'].append(val_mse_avg)
        history['train_swd'].append(train_swd_avg)
        history['val_swd'].append(val_swd_avg)

        scheduler.step(val_loss_avg)

        if val_loss_avg < best_val_loss:
            best_val_loss = val_loss_avg

            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss_avg,
                'val_loss': val_loss_avg,
                'lambda_swd': lambda_swd,
                'num_projections': num_projections,
                'tb_mean': processor.tb_mean.cpu(),
                'tb_std': processor.tb_std.cpu(),
                'era5_mean': processor.era5_mean.cpu(),
                'era5_std': processor.era5_std.cpu(),
                'rain_mean': processor.rain_mean.cpu() if torch.is_tensor(processor.rain_mean) else torch.tensor(processor.rain_mean),
                'rain_std': processor.rain_std.cpu() if torch.is_tensor(processor.rain_std) else torch.tensor(processor.rain_std),
                'zero_threshold': processor.zero_threshold
            }

            torch.save(checkpoint, BEST_MODEL_PATH)
            print(f"  ✓ Saved best model to {BEST_MODEL_PATH} (val_loss: {val_loss_avg:.6f})")

        print(
            f"Epoch {epoch+1:3d}/{num_epochs} | "
            f"Train Loss: {train_loss_avg:.6f} | Val Loss: {val_loss_avg:.6f} | "
            f"Train MSE: {train_mse_avg:.6f} | Val MSE: {val_mse_avg:.6f} | "
            f"Train SWD: {train_swd_avg:.6f} | Val SWD: {val_swd_avg:.6f} | "
            f"LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

    print("=" * 70)
    print(f"Training completed! Best validation loss: {best_val_loss:.6f}")

    return model, processor, history

## 12. Evaluation function
Inference runs in small GPU batches. MAE, RMSE, bias, and correlation use all pixels in each patch. POD/FAR use the configured event threshold. Constant-field correlation retains the original value of zero.

In [ ]:
def evaluate_resunet_model(model, processor, test_data, num_samples=150):
    tb_test, era5_test, rain_test = test_data

    print(f"Test data shapes:")
    print(f"  TB test: {tb_test.shape}")
    print(f"  ERA5 test: {era5_test.shape}")
    print(f"  Rain test: {rain_test.shape}")

    actual_samples = min(num_samples, rain_test.shape[0])
    if actual_samples < num_samples:
        print(f"Warning: Only {actual_samples} samples available, using {actual_samples} instead of {num_samples}")
        num_samples = actual_samples

    if num_samples == 0:
        print("Error: No test samples available!")
        return []

    device = next(model.parameters()).device
    model.eval()
    prediction_batches = []
    with torch.no_grad():
        for start in range(0, num_samples, INFERENCE_BATCH_SIZE):
            stop = min(start + INFERENCE_BATCH_SIZE, num_samples)
            tb_batch = processor.normalize_tb(tb_test[start:stop]).to(device)
            era5_batch = processor.normalize_era5(era5_test[start:stop]).to(device)
            pred_norm = model(tb_batch, era5_batch)
            prediction_batches.append(processor.denormalize_rain(pred_norm).cpu())
    predictions = torch.cat(prediction_batches, dim=0)

    predictions = predictions.cpu()
    rain_true = rain_test[:num_samples]

    save_dir = COMPARISON_DIR
    rain_cmap = cmap_rate_rain
    diff_cmap = plt.cm.RdBu_r

    all_metrics = []

    for i in range(num_samples):
        true_rain = rain_true[i].numpy()
        pred_rain = predictions[i].numpy()

        true_rain_2d = true_rain[0] if true_rain.ndim == 3 else true_rain
        pred_rain_2d = pred_rain[0] if pred_rain.ndim == 3 else pred_rain

        diff = pred_rain_2d - true_rain_2d
        mae = np.mean(np.abs(diff))
        rmse = np.sqrt(np.mean(diff ** 2))
        bias = np.mean(diff)

        flat_true = true_rain_2d.flatten()
        flat_pred = pred_rain_2d.flatten()
        if np.std(flat_true) > 0 and np.std(flat_pred) > 0:
            correlation = np.corrcoef(flat_true, flat_pred)[0, 1]
        else:
            correlation = 0.0

        zero_threshold = processor.zero_threshold
        zero_fraction_true = (true_rain_2d < zero_threshold).mean()
        zero_fraction_pred = (pred_rain_2d < zero_threshold).mean()

        true_rain_mask = true_rain_2d >= zero_threshold
        pred_rain_mask = pred_rain_2d >= zero_threshold

        if true_rain_mask.any():
            pod = np.sum(true_rain_mask & pred_rain_mask) / np.sum(true_rain_mask)
        else:
            pod = np.nan

        if pred_rain_mask.any():
            far = np.sum(pred_rain_mask & ~true_rain_mask) / np.sum(pred_rain_mask)
        else:
            far = np.nan

        metrics = {
            'true_sample': true_rain_2d,
            'pred_sample': pred_rain_2d,
            'mae': mae,
            'rmse': rmse,
            'bias': bias,
            'correlation': correlation,
            'zero_fraction_true': zero_fraction_true,
            'zero_fraction_pred': zero_fraction_pred,
            'pod': pod,
            'far': far,
            'sample_idx': i
        }
        all_metrics.append(metrics)

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        vmax_rain = 4.0

        im1 = axes[0].imshow(true_rain_2d, cmap=rain_cmap, vmin=0, vmax=vmax_rain)
        axes[0].set_title('True Rainfall', fontsize=12, fontweight='bold')
        axes[0].set_xticks([])
        axes[0].set_yticks([])
        plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04, label='Rain Rate (mm/hr)')

        im2 = axes[1].imshow(pred_rain_2d, cmap=rain_cmap, vmin=0, vmax=vmax_rain)
        axes[1].set_title(MODEL_LABEL, fontsize=12, fontweight='bold')
        axes[1].set_xlabel(f'MAE: {mae:.3f}, RMSE: {rmse:.3f}\nBias: {bias:.3f}',
                           fontsize=10, fontweight='bold')
        axes[1].set_xticks([])
        axes[1].set_yticks([])
        plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04, label='Rain Rate (mm/hr)')

        max_abs_diff = max(np.abs(diff).max(), 0.5)
        max_abs_diff = min(max_abs_diff, 2.0)

        im3 = axes[2].imshow(diff, cmap=diff_cmap, vmin=-max_abs_diff, vmax=max_abs_diff)
        axes[2].set_title('Difference (Pred - True)', fontsize=12, fontweight='bold')
        axes[2].set_xlabel(f'Correlation: {correlation:.3f}', fontsize=10, fontweight='bold')
        axes[2].set_xticks([])
        axes[2].set_yticks([])
        plt.colorbar(im3, ax=axes[2], fraction=0.046, pad=0.04, label='Difference (mm/hr)')

        plt.suptitle(f'Sample {i+1} - {MODEL_LABEL}', fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()

        save_path = os.path.join(save_dir, f'resunet_swd_sample_{i+1:02d}.png')
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close(fig)

    return all_metrics

## 13. Create and save the file split
The counts refer to candidate files before loader filtering. This cell stops if any requested split cannot be filled.

In [ ]:
if not PATCHES_DIR.is_dir():
    raise FileNotFoundError(f'Place the .npy patches in {PATCHES_DIR}, or edit PATCHES_DIR in cell 4.')
all_files = sorted(p.name for p in PATCHES_DIR.glob('*.npy'))
counts = [TRAIN_FILE_COUNT, VAL_FILE_COUNT, TEST_FILE_COUNT]
if any(n <= 0 for n in counts):
    raise ValueError('Each split count must be positive.')
if len(all_files) < sum(counts):
    raise ValueError(f'Found {len(all_files)} files; requested {sum(counts)}. Reduce counts in cell 4.')
rng = np.random.default_rng(SEED)
selected_files = [all_files[i] for i in rng.permutation(len(all_files))[:sum(counts)]]
train_files = selected_files[:TRAIN_FILE_COUNT]
val_files = selected_files[TRAIN_FILE_COUNT:TRAIN_FILE_COUNT + VAL_FILE_COUNT]
test_files = selected_files[TRAIN_FILE_COUNT + VAL_FILE_COUNT:]
with open(os.path.join(OUTPUT_DIR, 'candidate_file_splits.json'), 'w') as f:
    json.dump({'seed': SEED, 'train': train_files, 'validation': val_files, 'test': test_files}, f, indent=2)
print('Candidate split sizes:', len(train_files), len(val_files), len(test_files))

## 14. Load training data
This scans the complete training candidate list, then applies the custom selection.

In [ ]:
train_data = load_train_data_custom(
    patches_dir=str(PATCHES_DIR), file_list=train_files,
    low_no_sd_cap=LOW_NO_SD_CAP_TRAIN, high_no_sd_cap=HIGH_NO_SD_CAP_TRAIN,
    lsm_idx=LSM_IDX, lsm_threshold=LSM_THRESHOLD, sd_idx=SD_IDX, verbose_every=500,
)

## 15. Load validation and test data

In [ ]:
val_data = load_data_simple(str(PATCHES_DIR), file_list=val_files, verbose_every=200)
test_data = load_data_simple(str(PATCHES_DIR), file_list=test_files, verbose_every=200)
print('Loaded split sizes:', *(data[0].shape[0] for data in (train_data, val_data, test_data)))

## 16. Train
This cell starts optimization. Re-running it creates a newly initialized model.

In [ ]:
model, processor, history = train_resunet_model(
    train_data, val_data, num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, lambda_swd=LAMBDA_SWD,
    num_projections=NUM_PROJECTIONS,
)
with open(os.path.join(OUTPUT_DIR, 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)

## 17. Plot training history
Display and save total loss, MSE, and unweighted SWD separately.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key, title in zip(axes, ['loss', 'mse', 'swd'], ['Total loss', 'MSE', 'SWD (unweighted)']):
    epochs = np.arange(1, len(history['train_' + key]) + 1)
    ax.plot(epochs, history['train_' + key], label='Train')
    ax.plot(epochs, history['val_' + key], label='Validation')
    ax.set(xlabel='Epoch', ylabel=title, title=title)
    ax.legend()
    ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(HISTORY_PLOT_PATH, dpi=150, bbox_inches='tight')
plt.show()

## 19. Evaluate and export metrics
One image is saved per evaluated patch. The CSV contains scalar patch metrics; its sample index refers to the successfully loaded test order, which may differ from candidate order after skips.

In [ ]:
num_samples = test_data[0].shape[0] if EVALUATION_SAMPLES is None else EVALUATION_SAMPLES
all_metrics = evaluate_resunet_model(model, processor, test_data, num_samples=num_samples)
metric_keys = ['sample_idx', 'mae', 'rmse', 'bias', 'correlation', 'zero_fraction_true', 'zero_fraction_pred', 'pod', 'far']
metrics_path = os.path.join(OUTPUT_DIR, 'test_patch_metrics.csv')
with open(metrics_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=metric_keys)
    writer.writeheader()
    writer.writerows({key: row[key] for key in metric_keys} for row in all_metrics)
for key in ['mae', 'rmse', 'bias', 'correlation', 'pod', 'far']:
    values = np.array([row[key] for row in all_metrics], dtype=float)
    print(f'{key}: {np.nanmean(values):.6f}' if np.isfinite(values).any() else f'{key}: undefined')
print('Above values are means of patch metrics, not pooled-pixel RMSE.')
print('Outputs:', OUTPUT_DIR)

## 20. Optional: preview a saved comparison

In [ ]:
from IPython.display import display, Image
images = sorted(Path(COMPARISON_DIR).glob('*.png'))
if images:
    display(Image(filename=str(images[0])))